# RepurposeApp Video Worker — Free Google Colab T4 GPU Runner

This notebook runs the asynchronous video repurposing worker on a free Google Colab T4 GPU.
It polls jobs from your RepurposeApp dashboard, runs Whisper with CUDA acceleration (~10x faster than CPU), cuts 9:16 vertical promo reels with FFmpeg, and uploads the finished reels back to Cloudflare R2.

--- 

### 🔑 Setup: Add Your Secrets in Google Colab (One-time setup)
On the left sidebar, click the **🔑 Key icon (Secrets)** and add these keys with **Notebook access** enabled:

| Secret Name | Value |
| :--- | :--- |
| `NEXT_PUBLIC_SUPABASE_URL` | Your Supabase project URL |
| `SUPABASE_SERVICE_ROLE_KEY` | Your Supabase service role key |
| `R2_ACCOUNT_ID` | Your Cloudflare R2 account ID |
| `R2_ACCESS_KEY_ID` | Your Cloudflare R2 access key ID |
| `R2_SECRET_ACCESS_KEY` | Your Cloudflare R2 secret access key |
| `GEMINI_API_KEY` | (Optional) Your Google Gemini API key for AI hooks |

> **GPU Check:** Ensure hardware acceleration is enabled: `Runtime` -> `Change runtime type` -> select `T4 GPU`.

In [ ]:
# 1. Clone repository and install dependencies
!git clone -b feature/video-reels https://github.com/andreigrigore464-lang/RepurposeApp.git
%cd RepurposeApp/worker
!pip install -r requirements.txt

In [ ]:
# 2. Verify Colab Secrets (🔑 tab)
from google.colab import userdata

required_keys = [
    "NEXT_PUBLIC_SUPABASE_URL",
    "SUPABASE_SERVICE_ROLE_KEY",
    "R2_ACCOUNT_ID",
    "R2_ACCESS_KEY_ID",
    "R2_SECRET_ACCESS_KEY"
]

missing = []
for key in required_keys:
    try:
        val = userdata.get(key)
        if not val:
            missing.append(key)
        else:
            print(f"✓ {key}: configured")
    except Exception:
        missing.append(key)

if missing:
    print(f"\n⚠️ Missing {len(missing)} secret(s) in Colab Secrets tab (🔑): {missing}")
    print("Please add them in the left sidebar under the 🔑 icon and grant Notebook access.")
else:
    print("\n🚀 All required secrets verified! Ready to start worker.")

In [ ]:
# 3. Run the video worker daemon (listens for jobs live from your dashboard)
!python main.py